In [1]:
# @title
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [2]:
# @title
!pip -q install --upgrade transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.0 MB/s eta 0:00:00


In [3]:
# @title
import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [4]:
# @title
SYSTEM_PROMPT = """
You are a Senior Software Requirements Engineer working in a regulated compliance environment.

Your task is to create test cases including the input and expected output for converting regulatory legal text into precise, testable, system-level software requirements.

Rules:
1. Only extract explicit regulatory obligations.
2. Do NOT hallucinate new rules.
3. Do NOT interpret beyond the text.
4. Each requirement must begin with: "System shall".
5. Requirements must be atomic (one obligation per requirement).
6. Output MUST be valid JSON.
7. Do not include explanations.
8. Do not merge separate obligations.
9. Preserve time constraints exactly as written.
10. Identify role-based obligations explicitly (e.g., PCQI).

Some examples of correct output:
[
  {
    "test_case_id": "TC-001",
    "requirement_id": "REQ-117.130-001A",
    "input_data": "Conduct hazard analysis for each type of food."
    "expected_output": "Verify that a hazard analysis is conducted for each type of food."
  },
  {
    "test_case_id": "TC-002",
    "requirement_id": "REQ-117.130-001B",
    "input_data": "Hazard analysis must be documented."
    "expected_output": "Verify that the hazard analysis is documented."
  }
}

"""

In [5]:
# @title
def build_prompt(reg_text):
    return f"""
[INST] <<SYS>>
{SYSTEM_PROMPT}
<</SYS>>
Convert the following regulatory text into atomic system requirements in JSON ONLY:

{reg_text}
[/INST]
"""

In [6]:
# @title
#Make sure the model is loaded on GPU
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [11]:
# @title
def generate(model, text, max_new_tokens=800):
    prompt = build_prompt(text)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result

In [9]:
# @title
reg_text = """
# §117.130 Hazard Analysis – Atomic Rule Hierarchy

## (a) Requirement for a hazard analysis → REQ-117.130-001
- (1) Conduct hazard analysis → A
  - Identify known hazards → B
  - Identify reasonably foreseeable hazards → C
  - Evaluate hazards using data, experience, reports → D
  - Determine hazards requiring preventive controls → E
- (2) Hazard analysis must be written → F

## (b) Hazard identification → REQ-117.130-002
- (1) Known or reasonably foreseeable hazards include:
  - Biological hazards → A
  - Chemical hazards → B
  - Physical hazards → C
- (2) Consider hazards for origin:
  - Occur naturally → A
  - Unintentionally introduced → B
  - Intentionally introduced for economic gain → C

## (c) Hazard evaluation → REQ-117.130-003
- (1) Evaluate identified hazards → A
  - (i) Assess severity and probability of hazard occurrence → A1
  - (ii) Evaluate environmental pathogens for ready-to-eat foods → A2
- (2) Consider effects on safety of finished food → B
  - (i) Formulation of the food → B1
  - (ii) Facility and equipment condition, function, design → B2
  - (iii) Raw materials and other ingredients → B3
  - (iv) Transportation practices → B4
  - (v) Manufacturing/processing procedures → B5
  - (vi) Packaging and labeling activities → B6
  - (vii) Storage and distribution → B7
  - (viii) Intended or reasonably foreseeable use → B8
  - (ix) Sanitation, including employee hygiene → B9
  - (x) Any other relevant factors (e.g., temporal/natural toxins) → B10
"""

Load fp16 model

In [7]:
# @title
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Run fp16

In [ ]:
# @title
output_fp16 = generate(model_fp16, reg_text)
print("===== FP16 MODEL OUTPUT =====")
print(output_fp16)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Load 4bit model

In [ ]:
# @title
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

#got error for memory, need to unload last model and then clear cache to allow loading the next model
import gc

model_fp16 = None
gc.collect() # Python thing
torch.cuda.empty_cache() # PyTorch thing

#good to go probably

model_fp4 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    dtype="auto",
    quantization_config=quantization_config
)

Run fp4

In [ ]:
# @title
output_fp4 = generate(model_fp4, reg_text)
print("===== FP4 MODEL OUTPUT =====")
print(output_fp4)